In [1]:
import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

class RBM:
    def __init__(self, num_visible, num_hidden, learning_rate=0.1):
        self.num_visible = num_visible
        self.num_hidden = num_hidden
        self.lr = learning_rate
        
        # Initialize weights randomly (mean=0, std=0.1)
        self.W = np.random.normal(0, 0.1, (num_visible, num_hidden))
        # Biases
        self.b_v = np.zeros(num_visible) # Visible bias
        self.b_h = np.zeros(num_hidden)  # Hidden bias

    def sample_hidden(self, v_states):
        """Pass visible states up to hidden layer to get probabilities and sample"""
        h_activations = np.dot(v_states, self.W) + self.b_h
        h_probs = sigmoid(h_activations)
        # Stochastically turn neurons on/off based on probability
        h_states = (h_probs > np.random.rand(self.num_hidden)).astype(float)
        return h_probs, h_states

    def sample_visible(self, h_states):
        """Pass hidden states down to visible layer to get probabilities and sample"""
        v_activations = np.dot(h_states, self.W.T) + self.b_v
        v_probs = sigmoid(v_activations)
        v_states = (v_probs > np.random.rand(self.num_visible)).astype(float)
        return v_probs, v_states

    def train_step(self, v0_states):
        """Contrastive Divergence (CD-1) Learning Rule"""
        # --- POSITIVE PHASE (AWAKE) ---
        # Data is clamped to visible units. Calculate hidden units.
        h0_probs, h0_states = self.sample_hidden(v0_states)
        # Calculate positive associations <v_i * h_j>_data
        pos_associations = np.outer(v0_states, h0_probs) 

        # --- NEGATIVE PHASE (DREAMING) ---
        # Reconstruct the visible units from the hidden states (1 step of dreaming)
        v1_probs, v1_states = self.sample_visible(h0_states)
        # Recalculate hidden states from the reconstruction
        h1_probs, h1_states = self.sample_hidden(v1_states)
        # Calculate negative associations <v_i * h_j>_model
        neg_associations = np.outer(v1_probs, h1_probs)

        # --- UPDATE WEIGHTS & BIASES ---
        # Update Rule: dW = lr * (Awake - Dream)
        self.W += self.lr * (pos_associations - neg_associations)
        self.b_v += self.lr * (v0_states - v1_probs)
        self.b_h += self.lr * (h0_probs - h1_probs)
        
        # Calculate error for tracking
        error = np.mean((v0_states - v1_probs) ** 2)
        return error

# 1. Define Training Data (Left vs Right patterns)
training_data = np.array([
    [1, 1, 1, 0, 0, 0],
    [1, 0, 1, 0, 0, 0], # slightly noisy variant
    [1, 1, 0, 0, 0, 0], # slightly noisy variant
    [0, 0, 0, 1, 1, 1],
    [0, 0, 1, 1, 1, 0], # slightly noisy variant
    [0, 0, 0, 1, 0, 1]  # slightly noisy variant
])

# 2. Initialize RBM (6 visible inputs, 2 hidden feature detectors)
rbm = RBM(num_visible=6, num_hidden=2, learning_rate=0.1)

# 3. Train the Network
print("Training started...")
epochs = 1000
for epoch in range(epochs):
    total_error = 0
    for data_point in training_data:
        total_error += rbm.train_step(data_point)
    
    if epoch % 200 == 0:
        print(f"Epoch {epoch} | Reconstruction Error: {total_error / len(training_data):.4f}")

print("Training complete.\n")

# 4. Test: Reconstructing a corrupted pattern
noisy_input = np.array([1, 1, 0, 0, 0, 1]) # Mostly left, but one stray bit on the right
print(f"Original Noisy Input: {noisy_input}")

# Push it up to the hidden layer, then down to reconstruct
_, h_states = rbm.sample_hidden(noisy_input)
v_probs, _ = rbm.sample_visible(h_states)

# Round probabilities to nearest integer for clean output
reconstructed = np.round(v_probs).astype(int)
print(f"Reconstructed Output: {reconstructed}")

Training started...
Epoch 0 | Reconstruction Error: 0.2568
Epoch 200 | Reconstruction Error: 0.0968
Epoch 400 | Reconstruction Error: 0.0965
Epoch 600 | Reconstruction Error: 0.0964
Epoch 800 | Reconstruction Error: 0.0964
Training complete.

Original Noisy Input: [1 1 0 0 0 1]
Reconstructed Output: [1 1 1 0 0 0]
